# 실습 문제 - 104. SQL Agent

`# TODO` 표시된 부분을 채우며 위에서부터 셀 단위로 실행하세요. (모범답안: `practice_answer_104_SQL_Agent.ipynb`)

### 문제 요약

1. **다양한 질의 패턴 실행**: 4개의 질문(단순 집계 / 조인+순위 / 그룹별 집계 / 조인 조회)을
   리스트로 순회하며 SQL 에이전트로 실행하고 최종 답변 출력
2. **Human-in-the-Loop 승인 흐름 완성**: `HumanInTheLoopMiddleware` + `InMemorySaver` 에이전트로
   `sql_db_query` 실행 직전에 인터럽트 발생 확인 → `Command(resume=...)` 승인으로 최종 답변까지 실행

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 준비: 데이터베이스 연결 + SQL 도구 + 에이전트 (본문과 동일)

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_agent

model = init_chat_model(
    os.environ["OPENAI_MODEL"],
    model_provider="openai",
    use_responses_api=True,        # Responses API 사용 (Chat Completions에서는 추론 모델의 도구 호출 불가)
    reasoning={"effort": "low"},   # 추론 강도 낮춤 → 실습 속도·비용 절감
)

# SQLite 데이터베이스 연결
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

# SQL 도구 키트 생성
toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools = toolkit.get_tools()

# 시스템 프롬프트 설정 (본문과 동일)
system_prompt = """
당신은 SQL 데이터베이스와 상호작용하도록 설계된 에이전트입니다.
사용자로부터 질문을 입력받으면, 실행 가능한 {dialect} 구문으로
문법적으로 올바른 SQL 쿼리를 생성하세요.

그 후 쿼리를 실행한 결과를 확인하고, 그에 기반하여 답변을 반환하세요.
사용자가 명시적으로 원하는 예시 개수를 지정하지 않은 경우,
항상 최대 {top_k}개의 결과만 반환하도록 쿼리를 제한해야 합니다.

데이터베이스에서 가장 흥미로운 예시를 보여주기 위해,
관련 있는 열(column)을 기준으로 결과를 정렬할 수 있습니다.
특정 테이블의 모든 열을 조회하지 말고, 질문에 필요한 관련 열만 선택하세요.

쿼리를 실행하기 전에 반드시 구문을 두 번 확인하세요.
만약 쿼리 실행 중 오류가 발생하면, 쿼리를 다시 작성하고 재시도해야 합니다.

데이터베이스에 DML 문(INSERT, UPDATE, DELETE, DROP 등)을
절대 실행하지 마세요.

항상 데이터베이스에 어떤 테이블들이 존재하는지 먼저 확인한 뒤
어떤 데이터를 조회할 수 있는지 파악해야 합니다.
이 단계를 건너뛰지 마세요.

그 다음, 가장 관련성 높은 테이블의 스키마(schema)를 조회하세요.
""".format(
    dialect=db.dialect,
    top_k=5,
)

# SQL 에이전트 생성
agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
)

## 1. 다양한 질의 패턴 실행

In [ ]:
# 질의 패턴 테스트
queries = [
    "총 몇 명의 아티스트가 있나요?",          # 단순 집계
    # "가장 인기 있는 장르 상위 5개는?",        # 조인 + 순위
    # "고객별 총 구매 금액을 계산해주세요.",     # 그룹별 집계
    # "앨범과 아티스트 정보를 함께 보여주세요."  # 조인 조회
]

for query in queries:
    print(f"\n{'='*80}")
    print(f"질문: {query}")
    print("="*80)
    
    # TODO: agent 를 query 로 실행하세요. (messages 형식으로 전달)
    result = ...
    
    print("\n답변:")
    print(result["messages"][-1].text)

## 2. Human-in-the-Loop 승인 흐름 완성

### 2.1 검토용 에이전트 생성

`sql_db_query` 도구 호출 시 일시 정지하는 에이전트를 만듭니다.
인터럽트 후 재개하려면 상태 저장이 필요하므로 checkpointer가 필수입니다.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# Human-in-the-loop 미들웨어가 있는 에이전트 생성
agent_with_review = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            # TODO: sql_db_query 도구 호출 시 일시 정지하도록 지정하세요.
            interrupt_on=...,
            description_prefix="도구 실행 승인 대기 중",
        ),
    ],
    # TODO: 인터럽트 후 재개하려면 상태 저장이 필요합니다. 체크포인터를 지정하세요.
    checkpointer=...,
)

### 2.2 실행 -> 인터럽트 확인

에이전트가 SQL 쿼리를 실행하기 직전에 멈추고 승인을 기다립니다.

In [ ]:
question = "고객별 총 구매 금액을 계산해주세요."
config = {"configurable": {"thread_id": "practice_review_1"}}  # 새 thread_id 지정

print(f"질문: {question}\n")
print("=" * 80)

# 스트리밍 실행
for step in agent_with_review.stream(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    stream_mode="values",
):
    # 인터럽트 확인
    # TODO: step 에 인터럽트 키("__interrupt__")가 있는지 확인하세요.
    if ...:
        print("\n⚠️ HUMAN REVIEW REQUIRED ⚠️")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(f"\n대기 중인 작업:")
            print(f"설명: {request.get('description', 'N/A')}")
            if 'args' in request:
                print(f"실행할 쿼리: {request['args'].get('query', 'N/A')}")
        break
    elif "messages" in step:
        step["messages"][-1].pretty_print()

### 2.3 승인 -> 최종 답변까지 실행

쿼리를 검토한 후 `approve` 결정으로 재개하면, 멈췄던 지점부터 실행이 계속되어 최종 답변이 생성됩니다.
(같은 `thread_id`의 config를 전달해야 멈춘 대화가 이어집니다)

In [ ]:
from langgraph.types import Command

# 승인하여 실행 재개
result = agent_with_review.invoke(
    # TODO: 승인(approve) 결정을 resume 으로 전달하세요.
    Command(resume=...),
    config=config
)

print("승인 후 최종 답변:")
print(result["messages"][-1].text)